# Michigan Ave stop discovery — candidate routes

Fetch **all Bus Tracker `getpatterns` stops** for downtown Michigan Avenue candidate routes, then filter to the **Roosevelt → Delaware** corridor (vault `michigan-ave-bus-lane-segment`).

## How to run

1. Set `CTA_API_KEY` in the repo-root [`.env`](../.env) (see [`.env.example`](../.env.example)).
2. From the repo root:
   ```bash
   env/bin/jupyter lab notebooks/michigan_ave_stop_discovery.ipynb
   ```
3. **Run All** through §4 and review the corridor stop table. Set `WRITE_SNAPSHOT = True` in the setup cell only after §4 looks correct.

| Step | Source | Purpose |
|------|--------|--------|
| §1 | Mansueto `rt_to_pid.csv` + CloudFront | PIDs per route; parquet availability / trip counts |
| §2 | Bus Tracker API | Pattern stops per PID |
| §3 | Derived | Union catalog of all pattern stops |
| §4 | Derived | Michigan Ave corridor (Roosevelt–Delaware) |
| §5 | Local JSON | Optional `michigan_ave_stops_snapshot.json` |

Requires network access to CloudFront and `ctabustracker.com`.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd() if (Path.cwd() / ".env").is_file() else Path.cwd().parent
_nb = REPO_ROOT / "notebooks"
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))

from cta_api import (
    RT_TO_PID_URL,
    batch_pattern_stops,
    load_dotenv,
    parquet_http_status,
    parquet_url,
)

# User-provided Michigan Ave corridor candidates (system map + DLSD family).
CANDIDATE_ROUTES = [
    "1",
    "2",
    "3",
    "4",
    "X4",
    "6",
    "7",
    "10",
    "J14",
    "26",
    "28",
    "125",
    "126",
    "143",
    "146",
    "147",
    "148",
    "151",
    "157",
]

STOPS_SNAPSHOT_PATH = REPO_ROOT / "notebooks" / "michigan_ave_stops_snapshot.json"

# Set True only after §4 corridor table matches expected geography.
WRITE_SNAPSHOT = False

# Rough downtown Michigan latitude band (refine after §4 review).
LAT_CORRIDOR_SOUTH = 41.865
LAT_CORRIDOR_NORTH = 41.902
KNOWN_DELAWARE_STPIDS = {"1128", "1097"}

load_dotenv()
if not os.environ.get("CTA_API_KEY"):
    raise RuntimeError("Set CTA_API_KEY in .env at the repo root")


def resolve_routes_in_rt_df(rt_df: pd.DataFrame, candidates: list[str]) -> tuple[dict[str, list[str]], list[str]]:
    """Map candidate route labels to PIDs; flag candidates missing from rt_to_pid.csv."""
    rt_values = set(rt_df["rt"].astype(str))
    route_pids: dict[str, list[str]] = {}
    missing: list[str] = []
    for route in candidates:
        if route in rt_values:
            pids = rt_df.loc[rt_df["rt"] == route, "pid"].astype(str).tolist()
        elif route.upper() in rt_values:
            pids = rt_df.loc[rt_df["rt"] == route.upper(), "pid"].astype(str).tolist()
        else:
            missing.append(route)
            pids = []
        route_pids[route] = pids
    return route_pids, missing

## 1. PID discovery and Mansueto volume

List PIDs from `rt_to_pid.csv` for candidate routes; HEAD-check CloudFront parquets and count trips when available.

In [2]:
rt_df = pd.read_csv(RT_TO_PID_URL)
rt_df["rt"] = rt_df["rt"].astype(str)
rt_df["pid"] = rt_df["pid"].astype(str)

route_pids, missing_routes = resolve_routes_in_rt_df(rt_df, CANDIDATE_ROUTES)
if missing_routes:
    print("No rows in rt_to_pid.csv for:", missing_routes)

pid_to_route: dict[str, str] = {}
for route, pids in route_pids.items():
    for pid in pids:
        pid_to_route.setdefault(pid, route)

all_pids = sorted(pid_to_route.keys())
print(f"{len(CANDIDATE_ROUTES)} candidate routes → {len(all_pids)} distinct PIDs")

volume_rows = []
for route in CANDIDATE_ROUTES:
    for pid in route_pids[route]:
        status = parquet_http_status(pid)
        if status != 200:
            volume_rows.append(
                {
                    "route": route,
                    "pid": pid,
                    "trips": pd.NA,
                    "http_status": status,
                    "note": "missing on CloudFront",
                }
            )
            continue
        df = pd.read_parquet(parquet_url(pid), columns=["unique_trip_vehicle_day"])
        volume_rows.append(
            {
                "route": route,
                "pid": pid,
                "trips": df["unique_trip_vehicle_day"].nunique(),
                "http_status": 200,
                "note": None,
            }
        )
        del df

volume_df = pd.DataFrame(volume_rows)
display(volume_df.sort_values(["route", "pid"]))
print(f"PIDs with parquet on CloudFront: {(volume_df['http_status'] == 200).sum()} / {len(volume_df)}")

No rows in rt_to_pid.csv for: ['10']
19 candidate routes → 138 distinct PIDs


,route,pid,trips,http_status,note
0,1,6351,27358,200,NaN
1,1,8084,4245,200,NaN
2,1,8085,26172,200,NaN
69,125,2862,22833,200,NaN
68,125,2863,12531,200,NaN
...,...,...,...,...,...
48,J14,4278,876,200,NaN
49,J14,7173,1820,200,NaN
31,X4,19372,15601,200,NaN
33,X4,19373,1182,200,NaN


PIDs with parquet on CloudFront: 132 / 138


## 2. Fetch patterns (Bus Tracker API)

Call `getpatterns` once per PID (all PIDs from §1, not only those with Mansueto parquet).

In [3]:
all_pattern_stops: dict[str, tuple[str, pd.DataFrame]] = {}
pattern_meta_rows = []

for route in CANDIDATE_ROUTES:
    pids = route_pids[route]
    if not pids:
        continue
    print(f"Route {route}: {len(pids)} PIDs")
    batch = batch_pattern_stops(pids)
    all_pattern_stops.update(batch)
    for pid, (rtdir, stops) in batch.items():
        pattern_meta_rows.append(
            {
                "route": pid_to_route.get(pid, route),
                "pid": pid,
                "rtdir": rtdir,
                "pattern_stops": len(stops),
            }
        )

pattern_meta_df = pd.DataFrame(pattern_meta_rows).sort_values(["route", "pid"])
print(f"Loaded patterns for {len(all_pattern_stops)} PIDs")
display(pattern_meta_df)

Route 1: 3 PIDs
  Skipped 1 PIDs not in live API: ['8084']
  batch_pattern_stops done: live=2, cached=0, API today=2/10000
Route 2: 4 PIDs
  batch_pattern_stops done: live=4, cached=0, API today=6/10000
Route 3: 9 PIDs
  Skipped 1 PIDs not in live API: ['19388']
  batch_pattern_stops done: live=8, cached=0, API today=14/10000
Route 4: 15 PIDs
  Skipped 10 PIDs not in live API: ['2642', '7388', '20422', '10928', '19382', '19383', '19380', '19384', '8129', '19385']
  batch_pattern_stops done: live=5, cached=0, API today=19/10000
Route X4: 3 PIDs
  batch_pattern_stops done: live=3, cached=0, API today=22/10000
Route 6: 2 PIDs
  batch_pattern_stops done: live=2, cached=0, API today=24/10000
Route 7: 8 PIDs
  Skipped 2 PIDs not in live API: ['3626', '3761']
  batch_pattern_stops done: live=6, cached=0, API today=30/10000
Route J14: 6 PIDs
  Skipped 3 PIDs not in live API: ['4274', '4278', '7173']
  batch_pattern_stops done: live=3, cached=0, API today=33/10000
Route 26: 3 PIDs
  batch_patte

,route,pid,rtdir,pattern_stops
0,1,6351,Southbound,34
1,1,8085,Northbound,33
49,125,2862,Southbound,15
48,125,2863,Northbound,16
47,125,6369,Northbound,15
...,...,...,...,...
32,J14,4276,Northbound,33
31,J14,4277,Northbound,33
19,X4,19372,Northbound,36
21,X4,19373,Northbound,30


## 3. All stops catalog

Union every pattern stop; deduplicate by `stpid` for a master stop list.

In [4]:
catalog_rows = []
for pid, (rtdir, stops) in all_pattern_stops.items():
    route = pid_to_route.get(pid, "?")
    for _, row in stops.iterrows():
        catalog_rows.append(
            {
                "route": route,
                "pid": pid,
                "rtdir": rtdir,
                "seq": int(row["seq"]),
                "stpid": row["stpid"],
                "stpnm": row["stpnm"],
                "lat": row["lat"],
                "lon": row["lon"],
                "pdist_ft": row["pdist_ft"],
            }
        )

catalog_df = pd.DataFrame(catalog_rows)
print(f"Catalog rows (pattern × stop): {len(catalog_df)}")

def _join_sorted(values: pd.Series) -> str:
    return ",".join(sorted({str(v) for v in values.dropna()}))

stops_master = (
    catalog_df.groupby(["stpid", "stpnm", "lat", "lon"], as_index=False)
    .agg(
        routes=("route", _join_sorted),
        rtdirs=("rtdir", _join_sorted),
        n_patterns=("pid", "nunique"),
    )
    .sort_values(["lat", "lon", "stpid"])
    .reset_index(drop=True)
)
print(f"Unique stops: {len(stops_master)}")
display(stops_master.head(20))

Catalog rows (pattern × stop): 4794
Unique stops: 1380


,stpid,stpnm,lat,lon,routes,rtdirs,n_patterns
0,18241,115th Street Metra Station (Metra Electric),41.685671,-87.612446,4,"Northbound,Southbound",4
1,2841,Cottage Grove & 115th Street,41.685738,-87.611477,4,Southbound,2
2,2840,Cottage Grove & 114th Street,41.687320,-87.611143,4,Southbound,2
3,15156,Cottage Grove & 113th Street,41.688894,-87.610828,4,Southbound,2
4,14050,King Drive & 113th Street,41.688980,-87.613193,4,Northbound,2
5,14051,King Drive & 112th Street,41.690825,-87.613225,4,Northbound,2
6,2838,Cottage Grove & 112th Street,41.690848,-87.610317,4,Southbound,2
7,14052,King Drive & 111th Street,41.692588,-87.613273,4,Northbound,2
8,2893,111th Street & Vernon,41.692715,-87.612303,4,Northbound,2
9,2837,Cottage Grove & 111th Street,41.692930,-87.609887,4,Southbound,2


## 4. Michigan Ave corridor (Roosevelt → Delaware)

Filter stops on Michigan Avenue in the downtown band. Refine `LAT_CORRIDOR_*` in the setup cell if needed.

Delaware anchors (DLSD): **`1128`** (NB), **`1097`** (SB).

In [13]:
michigan_mask = catalog_df["stpnm"].astype(str).str.contains("Michigan", case=False, na=False)
lat_mask = catalog_df["lat"].between(LAT_CORRIDOR_SOUTH, LAT_CORRIDOR_NORTH)
delaware_mask = catalog_df["stpid"].isin(KNOWN_DELAWARE_STPIDS)

corridor_catalog = catalog_df.loc[(michigan_mask & lat_mask) | delaware_mask].copy()

corridor_master = (
    corridor_catalog.groupby(["stpid", "stpnm", "lat", "lon"], as_index=False)
    .agg(
        routes=("route", _join_sorted),
        rtdirs=("rtdir", _join_sorted),
        n_patterns=("pid", "nunique"),
    )
    .sort_values("lat", ascending=True)
    .reset_index(drop=True)
)

print(f"Corridor Michigan stops (unique): {len(corridor_master)}")
display(corridor_master)

roosevelt_candidates = corridor_master[
    corridor_master["stpnm"].astype(str).str.contains("Roosevelt", case=False, na=False)
]
print("\nRoosevelt-named stops in corridor filter (candidate south anchors):")
display(roosevelt_candidates)

Corridor Michigan stops (unique): 51


,stpid,stpnm,lat,lon,routes,rtdirs,n_patterns
0,1592,Michigan & 13th Street,41.865511,-87.624232,"1,3,4",Southbound,4
1,1579,Michigan & 13th Street,41.866121,-87.624011,"1,3,4",Northbound,6
2,1591,Michigan & Roosevelt,41.866781,-87.624256,"1,3,4,X4",Southbound,5
3,15362,Roosevelt & Michigan/Indiana,41.867373,-87.623385,146,Southbound,2
4,17084,Roosevelt & Michigan,41.867518,-87.624836,146,Northbound,4
5,1580,Michigan & Roosevelt,41.867930,-87.624018,"1,3,4,X4",Northbound,8
6,1590,Michigan & 11th Street,41.868891,-87.624240,"1,3,4,X4",Southbound,5
7,1581,Michigan & 11th Street,41.869098,-87.623970,"1,3,4,X4",Northbound,8
8,1589,Michigan & 9th Street,41.870345,-87.624273,"1,3,4,X4",Southbound,5
9,1582,Michigan & 9th Street,41.870393,-87.623987,"1,3,4,X4",Northbound,8



Roosevelt-named stops in corridor filter (candidate south anchors):


,stpid,stpnm,lat,lon,routes,rtdirs,n_patterns
2,1591,Michigan & Roosevelt,41.866781,-87.624256,"1,3,4,X4",Southbound,5
3,15362,Roosevelt & Michigan/Indiana,41.867373,-87.623385,146,Southbound,2
4,17084,Roosevelt & Michigan,41.867518,-87.624836,146,Northbound,4
5,1580,Michigan & Roosevelt,41.867930,-87.624018,"1,3,4,X4",Northbound,8


In [14]:
# Side-street stops at Michigan intersections (not on Michigan Ave itself).
# Also, 13th St is south of Roosevelt, and Oak is north of Delaware, so exclude them..
EXCLUDE_CORRIDOR_STPIDS: set[str] = {
    "1592", "1579", "15362", "17084", "4870", "4884", "74", "4725", "14482", "1421", "1129"
}

exclude = {str(s) for s in EXCLUDE_CORRIDOR_STPIDS}
stpid_str = corridor_master["stpid"].astype(str)

removed = corridor_master.loc[stpid_str.isin(exclude)].sort_values("lat")
if not removed.empty:
    print(f"Removing {len(removed)} side-street stop(s):")
    display(removed[["stpid", "stpnm", "lat", "lon", "routes", "rtdirs"]])
else:
    print("No rows matched EXCLUDE_CORRIDOR_STPIDS (check stpid strings).")

n_before = len(corridor_master)
corridor_master = (
    corridor_master.loc[~stpid_str.isin(exclude)]
    .sort_values("lat", ascending=True)
    .reset_index(drop=True)
)
print(f"corridor_master: {n_before} → {len(corridor_master)} stops")

Removing 11 side-street stop(s):


,stpid,stpnm,lat,lon,routes,rtdirs
0,1592,Michigan & 13th Street,41.865511,-87.624232,"1,3,4",Southbound
1,1579,Michigan & 13th Street,41.866121,-87.624011,"1,3,4",Northbound
3,15362,Roosevelt & Michigan/Indiana,41.867373,-87.623385,146,Southbound
4,17084,Roosevelt & Michigan,41.867518,-87.624836,146,Northbound
12,4870,Balbo & Michigan,41.873092,-87.623664,"2,26,28,6,J14",Southbound
13,4884,Balbo & Michigan,41.873239,-87.623810,"2,26,28,6,J14",Northbound
17,74,Congress Plaza & Michigan,41.874809,-87.623680,"126,143,147,7","Eastbound,Northbound,Southbound,Westbound"
19,4725,Ida B Wells Drive & Michigan,41.875858,-87.623946,"126,143,147,7","Northbound,Westbound"
28,14482,Randolph & Michigan,41.884365,-87.623715,"4,X4",Northbound
35,1421,Wacker (Upper) & Michigan,41.888371,-87.625030,"146,2,6",Southbound


corridor_master: 51 → 40 stops


In [15]:
corridor_master.sort_values("lat")

,stpid,stpnm,lat,lon,routes,rtdirs,n_patterns
0,1591,Michigan & Roosevelt,41.866781,-87.624256,"1,3,4,X4",Southbound,5
1,1580,Michigan & Roosevelt,41.867930,-87.624018,"1,3,4,X4",Northbound,8
2,1590,Michigan & 11th Street,41.868891,-87.624240,"1,3,4,X4",Southbound,5
3,1581,Michigan & 11th Street,41.869098,-87.623970,"1,3,4,X4",Northbound,8
4,1589,Michigan & 9th Street,41.870345,-87.624273,"1,3,4,X4",Southbound,5
5,1582,Michigan & 9th Street,41.870393,-87.623987,"1,3,4,X4",Northbound,8
6,1583,Michigan & 8th Street,41.871728,-87.624018,"1,3,4,X4",Northbound,8
7,1588,Michigan & 8th Street,41.872029,-87.624317,"1,3,4,X4",Southbound,5
8,1584,Michigan & Balbo,41.873444,-87.624107,"1,3,4,X4",Northbound,8
9,1587,Michigan & Balbo,41.873760,-87.624352,"1,3,4,X4",Southbound,5


## 5. Save snapshot (optional)

Writes `notebooks/michigan_ave_stops_snapshot.json` when `WRITE_SNAPSHOT = True`.

In [16]:
WRITE_SNAPSHOT = True

snapshot = {
    "candidate_routes": CANDIDATE_ROUTES,
    "corridor": {
        "description": "Michigan Ave downtown Roosevelt to Delaware",
        "lat_south": LAT_CORRIDOR_SOUTH,
        "lat_north": LAT_CORRIDOR_NORTH,
        "delaware_stpids": sorted(KNOWN_DELAWARE_STPIDS),
        "roosevelt_stpids": roosevelt_candidates["stpid"].astype(str).tolist(),
    },
    "stops": corridor_master.assign(stpid=corridor_master["stpid"].astype(str)).to_dict(orient="records"),
    "patterns_fetched": len(all_pattern_stops),
}

if not WRITE_SNAPSHOT:
    print(
        f"WRITE_SNAPSHOT is False — would write {len(snapshot['stops'])} corridor stops to "
        f"{STOPS_SNAPSHOT_PATH.name}. Set WRITE_SNAPSHOT = True after §4 review."
    )
else:
    STOPS_SNAPSHOT_PATH.write_text(json.dumps(snapshot, indent=2) + "\n")
    print(f"Wrote {STOPS_SNAPSHOT_PATH}")

snapshot["stops"][:3]

Wrote /Users/satoru/Documents/Documents - MacBook Pro (5)/Github/cta_bus_analysis/notebooks/michigan_ave_stops_snapshot.json


[{'stpid': '1591',
  'stpnm': 'Michigan & Roosevelt',
  'lat': 41.866781,
  'lon': -87.624256000001,
  'routes': '1,3,4,X4',
  'rtdirs': 'Southbound',
  'n_patterns': 5},
 {'stpid': '1580',
  'stpnm': 'Michigan & Roosevelt',
  'lat': 41.867930000001,
  'lon': -87.624018,
  'routes': '1,3,4,X4',
  'rtdirs': 'Northbound',
  'n_patterns': 8},
 {'stpid': '1590',
  'stpnm': 'Michigan & 11th Street',
  'lat': 41.868891000001,
  'lon': -87.62424,
  'routes': '1,3,4,X4',
  'rtdirs': 'Southbound',
  'n_patterns': 5}]